### Part A - Scikit-learn Implementation

In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("garments_worker_productivity.csv")

print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.describe())

(1197, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   object 
 1   quarter                1197 non-null   object 
 2   department             1197 non-null   object 
 3   day                    1197 non-null   object 
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float6

In [15]:
# create the classification target
df['MeetsTarget'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)

In [16]:
# define the feature sets for each task
# Drop non-predictive 
drop_cols = ['date']  # date is often just an identifier, not a real predictor here

# Regression: target = actual_productivity
reg_features = df.drop(columns=['actual_productivity', 'MeetsTarget'] + drop_cols)
reg_target = df['actual_productivity']

# Classification: target = MeetsTarget, and actual_productivity must NOT be a feature
clf_features = df.drop(columns=['actual_productivity', 'MeetsTarget'] + drop_cols)
clf_target = df['MeetsTarget']

In [17]:
# identify numerical and categorical columns
categorical_cols = reg_features.select_dtypes(include='object').columns.tolist()
numeric_cols = reg_features.select_dtypes(include=np.number).columns.tolist()

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['quarter', 'department', 'day']
Numeric: ['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers']


In [18]:
# build the preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline(steps= [
    ('impute', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps= [
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [19]:
# train-test split
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    reg_features, reg_target, test_size=0.2, random_state=42
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    clf_features, clf_target, test_size=0.2, random_state=42
)

In [20]:
# capture the indices to reuse in the part B
train_idx = X_train_reg.index
test_idx = X_test_reg.index

In [21]:
# train linear regression
import time
from sklearn.linear_model import LinearRegression

reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression()) 
])

start = time.perf_counter()
reg_pipeline.fit(X_train_reg, y_train_reg)
reg_train_time = time.perf_counter() - start

start = time.perf_counter()
y_pred_reg = reg_pipeline.predict(X_test_reg)
reg_predict_time = time.perf_counter() - start

print(f"Train time: {reg_train_time:.4f}s, Predict time: {reg_predict_time:.4f}s")

Train time: 0.0459s, Predict time: 0.0114s


In [50]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_sklearn = mean_absolute_error(y_test_reg, y_pred_reg)
rmse_sklearn = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2_sklearn = r2_score(y_test_reg, y_pred_reg)

print(mae_sklearn, rmse_sklearn, r2_sklearn)

0.10845261337977505 0.1486175780959365 0.168168256630562


In [23]:
# train logistic regression
from sklearn.linear_model import LogisticRegression

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

start = time.perf_counter()
clf_pipeline.fit(X_train_clf, y_train_clf)
clf_train_time = time.perf_counter() - start

start = time.perf_counter()
y_pred_clf = clf_pipeline.predict(X_test_clf)
clf_predict_time = time.perf_counter() - start

print(f"Train time: {clf_train_time:.4f}s, Predict time: {clf_predict_time:.4f}s")

Train time: 0.0797s, Predict time: 0.0178s


In [53]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_sklearn = accuracy_score(y_test_clf, y_pred_clf)
precision_sklearn = precision_score(y_test_clf, y_pred_clf)
recall_sklearn = recall_score(y_test_clf, y_pred_clf)
f1_sklearn = f1_score(y_test_clf, y_pred_clf)

print(accuracy_sklearn,precision_sklearn,recall_sklearn,f1_sklearn)

0.775 0.7808219178082192 0.9661016949152542 0.8636363636363636


In [25]:
results = pd.DataFrame({
    'Model': ['LinearRegression', 'LogisticRegression'],
    'Train Time (s)': [reg_train_time, clf_train_time],
    'Predict Time (s)': [reg_predict_time, clf_predict_time],
})

print(results)

print("\nRegression metrics:", {'MAE': mae, 'RMSE': rmse, 'R2': r2})
print("Classification metrics:", {
    'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1
})

                Model  Train Time (s)  Predict Time (s)
0    LinearRegression        0.045898          0.011446
1  LogisticRegression        0.079730          0.017794

Regression metrics: {'MAE': 0.10845261337977505, 'RMSE': np.float64(0.1486175780959365), 'R2': 0.168168256630562}
Classification metrics: {'Accuracy': 0.775, 'Precision': 0.7808219178082192, 'Recall': 0.9661016949152542, 'F1': 0.8636363636363636}


### Part B - From-Scratch Implementation

In [26]:
# load the dataset and slice the train test with captured indices from part A's split
df_raw = pd.read_csv("garments_worker_productivity.csv")
df_raw['MeetsTarget'] = (df_raw['actual_productivity'] >= df_raw['targeted_productivity']).astype(int)

train_df = df_raw.loc[train_idx].copy()
test_df = df_raw.loc[test_idx].copy()

In [27]:
# manual missing value imputation
numeric_cols = ['team', 'targeted_productivity', 'smv', 'wip', 'over_time',
                 'incentive', 'idle_time', 'idle_men', 'no_of_style_change',
                 'no_of_workers'] 
categorical_cols = ['department', 'quarter', 'day']  

# Compute imputation values from TRAIN data only
medians = train_df[numeric_cols].median()
modes = train_df[categorical_cols].mode().iloc[0]

# Apply to both train and test using train-derived values
train_df[numeric_cols] = train_df[numeric_cols].fillna(medians)
test_df[numeric_cols] = test_df[numeric_cols].fillna(medians)

train_df[categorical_cols] = train_df[categorical_cols].fillna(modes)
test_df[categorical_cols] = test_df[categorical_cols].fillna(modes)

In [28]:
# manual one-hot encoding
def manual_one_hot(train_df, test_df, columns):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    for col in columns:
        categories = sorted(train_df[col].unique())  # categories seen in TRAINING data
        for cat in categories:
            train_encoded[f"{col}_{cat}"] = (train_df[col] == cat).astype(int)
            test_encoded[f"{col}_{cat}"] = (test_df[col] == cat).astype(int)
        train_encoded.drop(columns=[col], inplace=True)
        test_encoded.drop(columns=[col], inplace=True)

    return train_encoded, test_encoded

train_df, test_df = manual_one_hot(train_df, test_df, categorical_cols)

In [29]:
# feature scaling
def manual_standard_scale(train_df, test_df, columns):
    means = train_df[columns].mean()
    stds = train_df[columns].std(ddof=0)  # population std, matches sklearn's default

    train_scaled = train_df.copy()
    test_scaled = test_df.copy()

    train_scaled[columns] = (train_df[columns] - means) / stds
    test_scaled[columns] = (test_df[columns] - means) / stds

    return train_scaled, test_scaled

train_df, test_df = manual_standard_scale(train_df, test_df, numeric_cols)

In [46]:
# prepare the final numpy array# Build feature matrices (drop target columns, keep everything else as features)
feature_names = train_df.drop(columns=['actual_productivity', 'MeetsTarget', 'date']).columns
X_train_reg = train_df.drop(columns=['actual_productivity', 'MeetsTarget', 'date']).values.astype(float)
X_test_reg = test_df.drop(columns=['actual_productivity', 'MeetsTarget', 'date']).values.astype(float)
y_train_reg = train_df['actual_productivity'].values.astype(float)
y_test_reg = test_df['actual_productivity'].values.astype(float)

# same features, classification just uses a different target
X_test_clf = X_test_reg.copy()
X_train_clf = X_train_reg.copy() 
y_train_clf = train_df['MeetsTarget'].values.astype(float)
y_test_clf = test_df['MeetsTarget'].values.astype(float)

# Add a column of 1s so the model can learn an intercept (bias) term
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

X_train_reg_b = add_bias(X_train_reg)
X_test_reg_b = add_bias(X_test_reg)
X_train_clf_b = add_bias(X_train_clf)
X_test_clf_b = add_bias(X_test_clf)

In [34]:
# linear regression - closed form solution
import time

def fit_linear_regression_normal_eq(X, y):
    # w = (X^T X)^-1 X^T y
    XtX = X.T @ X
    XtX_inv = np.linalg.pinv(XtX)
    Xty = X.T @ y
    w = XtX_inv @ Xty
    return w

start = time.perf_counter()
w_reg = fit_linear_regression_normal_eq(X_train_reg_b, y_train_reg)
reg_train_time_scratch = time.perf_counter() - start

start = time.perf_counter()
y_pred_reg_scratch = X_test_reg_b @ w_reg
reg_predict_time_scratch = time.perf_counter() - start

In [35]:
# regression metrics
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

mae_val = mae(y_test_reg, y_pred_reg_scratch)
rmse_val = rmse(y_test_reg, y_pred_reg_scratch)
r2_val = r2(y_test_reg, y_pred_reg_scratch)

In [36]:
# logistic regression - sigmoid function and gradient descent
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def fit_logistic_regression(X, y, lr=0.1, n_iters=1000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)  # start all weights at zero

    for i in range(n_iters):
        z = X @ w                     # linear combination
        p = sigmoid(z)                # predicted probabilities
        gradient = (X.T @ (p - y)) / n_samples   # vectorized gradient of log loss
        w -= lr * gradient             # update weights, step downhill

    return w

start = time.perf_counter()
w_clf = fit_logistic_regression(X_train_clf_b, y_train_clf, lr=0.1, n_iters=2000)
clf_train_time_scratch = time.perf_counter() - start

start = time.perf_counter()
probs_test = sigmoid(X_test_clf_b @ w_clf)
y_pred_clf_scratch = (probs_test >= 0.5).astype(int)
clf_predict_time_scratch = time.perf_counter() - start

In [37]:
# classification matrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

acc_val = accuracy(y_test_clf, y_pred_clf_scratch)
prec_val = precision(y_test_clf, y_pred_clf_scratch)
rec_val = recall(y_test_clf, y_pred_clf_scratch)
f1_val = f1(y_test_clf, y_pred_clf_scratch)

In [38]:
scratch_results = {
    'Regression': {'MAE': mae_val, 'RMSE': rmse_val, 'R2': r2_val,
                   'Train Time (s)': reg_train_time_scratch,
                   'Predict Time (s)': reg_predict_time_scratch},
    'Classification': {'Accuracy': acc_val, 'Precision': prec_val,
                        'Recall': rec_val, 'F1': f1_val,
                        'Train Time (s)': clf_train_time_scratch,
                        'Predict Time (s)': clf_predict_time_scratch}
}

print(scratch_results)

{'Regression': {'MAE': np.float64(0.108452613379775), 'RMSE': np.float64(0.14861757809593648), 'R2': np.float64(0.16816825663056223), 'Train Time (s)': 0.010794700006954372, 'Predict Time (s)': 0.0006593000143766403}, 'Classification': {'Accuracy': np.float64(0.775), 'Precision': np.float64(0.7808219178082192), 'Recall': np.float64(0.9661016949152542), 'F1': np.float64(0.8636363636363635), 'Train Time (s)': 0.15077420003945008, 'Predict Time (s)': 0.0001993000041693449}}


### Part C - Comparison and Optimization

In [54]:
import pandas as pd

regression_comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2', 'Train Time (s)', 'Predict Time (s)'],
    'Scikit-learn': [mae_sklearn, rmse_sklearn, r2_sklearn, reg_train_time, reg_predict_time],
    'Manual (NumPy)': [mae_val, rmse_val, r2_val, reg_train_time_scratch, reg_predict_time_scratch]
})

classification_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1', 'Train Time (s)', 'Predict Time (s)'],
    'Scikit-learn': [accuracy_sklearn, precision_sklearn, recall_sklearn, f1_sklearn, clf_train_time, clf_predict_time],
    'Manual (NumPy)': [acc_val, prec_val, rec_val, f1_val, clf_train_time_scratch, clf_predict_time_scratch]
})

print(regression_comparison)
print(classification_comparison)

             Metric  Scikit-learn  Manual (NumPy)
0               MAE      0.108453        0.108453
1              RMSE      0.148618        0.148618
2                R2      0.168168        0.168168
3    Train Time (s)      0.045898        0.010795
4  Predict Time (s)      0.011446        0.000659
             Metric  Scikit-learn  Manual (NumPy)
0          Accuracy      0.775000        0.775000
1         Precision      0.780822        0.780822
2            Recall      0.966102        0.966102
3                F1      0.863636        0.863636
4    Train Time (s)      0.079730        0.150774
5  Predict Time (s)      0.017794        0.000199


In [41]:
# optimization 1 - convergence tracking
def fit_logistic_regression_v2(X, y, lr=0.1, n_iters=5000, tol=1e-6):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    prev_cost = np.inf

    for i in range(n_iters):
        z = X @ w
        p = sigmoid(z)
        # log loss (cross-entropy), clipped to avoid log(0)
        p_clipped = np.clip(p, 1e-12, 1 - 1e-12)
        cost = -np.mean(y * np.log(p_clipped) + (1 - y) * np.log(1 - p_clipped))

        if abs(prev_cost - cost) < tol:
            print(f"Converged at iteration {i}")
            break
        prev_cost = cost

        gradient = (X.T @ (p - y)) / n_samples
        w -= lr * gradient

    return w

In [42]:
# optimization 2 - learning rate tuning
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]

for lr in learning_rates:
    w_test = fit_logistic_regression_v2(X_train_clf_b, y_train_clf, lr=lr, n_iters=5000)
    probs = sigmoid(X_test_clf_b @ w_test)
    preds = (probs >= 0.5).astype(int)
    acc = accuracy(y_test_clf, preds)
    print(f"lr={lr}: accuracy={acc:.4f}")

lr=0.001: accuracy=0.7417
lr=0.01: accuracy=0.7500
Converged at iteration 2129
lr=0.1: accuracy=0.7792
Converged at iteration 710
lr=0.5: accuracy=0.7750
Converged at iteration 435
lr=1.0: accuracy=0.7750


In [43]:
# optimization 3 - L2 Regularization
# for logistic regression
def fit_logistic_regression_regularized(X, y, lr=0.1, n_iters=5000, lam=1.0, tol=1e-6):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    prev_cost = np.inf

    for i in range(n_iters):
        z = X @ w
        p = sigmoid(z)
        p_clipped = np.clip(p, 1e-12, 1 - 1e-12)

        # regularization term excludes the bias weight w[0]
        reg_term = (lam / (2 * n_samples)) * np.sum(w[1:] ** 2)
        cost = -np.mean(y * np.log(p_clipped) + (1 - y) * np.log(1 - p_clipped)) + reg_term

        if abs(prev_cost - cost) < tol:
            break
        prev_cost = cost

        gradient = (X.T @ (p - y)) / n_samples
        gradient[1:] += (lam / n_samples) * w[1:]   # don't regularize the bias term
        w -= lr * gradient

    return w

In [44]:
# for linear regression
def fit_ridge_regression_normal_eq(X, y, lam=1.0):
    n_features = X.shape[1]
    I = np.eye(n_features)
    I[0, 0] = 0  # don't regularize the bias term
    w = np.linalg.inv(X.T @ X + lam * I) @ (X.T @ y)
    return w

In [48]:
# optimization 4 - feature selection
# Correlation-based feature selection for regression
feature_names = train_df.drop(columns=['actual_productivity', 'MeetsTarget','date']).columns
correlations = pd.Series(
    [np.corrcoef(X_train_reg[:, i], y_train_reg)[0, 1] for i in range(X_train_reg.shape[1])],
    index=feature_names
).abs().sort_values(ascending=False)

print(correlations)

# Keep only features above a correlation threshold
selected_features = correlations[correlations > 0.05].index.tolist()
print(f"Keeping {len(selected_features)} of {len(feature_names)} features")

targeted_productivity    0.431037
no_of_style_change       0.212743
idle_men                 0.202731
team                     0.170332
department_finishing     0.148015
smv                      0.117555
quarter_Quarter5         0.114409
day_Saturday             0.090156
idle_time                0.086370
department_sweing        0.081632
quarter_Quarter4         0.081149
wip                      0.079886
quarter_Quarter3         0.079648
quarter_Quarter1         0.076542
incentive                0.066243
no_of_workers            0.057404
over_time                0.051634
day_Thursday             0.051445
department_finishing     0.050359
day_Wednesday            0.046398
day_Tuesday              0.022896
quarter_Quarter2         0.012866
day_Sunday               0.007860
day_Monday               0.003148
dtype: float64
Keeping 19 of 24 features


In [55]:
# Retrain using regularization + tuned learning rate + convergence check
best_lr = 0.1  # replace with whichever value performed best in Step 4
w_reg_v2 = fit_ridge_regression_normal_eq(X_train_reg_b, y_train_reg, lam=1.0)
w_clf_v2 = fit_logistic_regression_regularized(X_train_clf_b, y_train_clf, lr=best_lr, n_iters=5000, lam=1.0)

y_pred_reg_v2 = X_test_reg_b @ w_reg_v2
probs_v2 = sigmoid(X_test_clf_b @ w_clf_v2)
y_pred_clf_v2 = (probs_v2 >= 0.5).astype(int)

final_comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2'],
    'Scikit-learn': [mae_sklearn, rmse_sklearn, r2_sklearn],
    'Manual v1 (unoptimized)': [mae_val, rmse_val, r2_val],
    'Manual v2 (optimized)': [
        mae(y_test_reg, y_pred_reg_v2),
        rmse(y_test_reg, y_pred_reg_v2),
        r2(y_test_reg, y_pred_reg_v2)
    ]
})
print(final_comparison)

  Metric  Scikit-learn  Manual v1 (unoptimized)  Manual v2 (optimized)
0    MAE      0.108453                 0.108453               0.108312
1   RMSE      0.148618                 0.148618               0.148501
2     R2      0.168168                 0.168168               0.169474


In [56]:
final_classification_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1'],
    'Scikit-learn': [accuracy_sklearn, precision_sklearn, recall_sklearn, f1_sklearn],
    'Manual v1 (unoptimized)': [acc_val, prec_val, rec_val, f1_val],
    'Manual v2 (optimized)': [
        accuracy(y_test_clf, y_pred_clf_v2),
        precision(y_test_clf, y_pred_clf_v2),
        recall(y_test_clf, y_pred_clf_v2),
        f1(y_test_clf, y_pred_clf_v2)
    ]
})
print(final_classification_comparison)

      Metric  Scikit-learn  Manual v1 (unoptimized)  Manual v2 (optimized)
0   Accuracy      0.775000                 0.775000               0.775000
1  Precision      0.780822                 0.780822               0.780822
2     Recall      0.966102                 0.966102               0.966102
3         F1      0.863636                 0.863636               0.863636


In [57]:
probs_v1 = sigmoid(X_test_clf_b @ w_clf)      # unregularized manual weights
probs_v2 = sigmoid(X_test_clf_b @ w_clf_v2)    # regularized manual weights

print("Max absolute difference in probabilities:", np.max(np.abs(probs_v1 - probs_v2)))
print("Number of predictions that flipped:", np.sum((probs_v1 >= 0.5).astype(int) != (probs_v2 >= 0.5).astype(int)))

print("\nSample weight comparison (first 5 weights):")
print("w_clf   (v1):", w_clf[:5])
print("w_clf_v2 (v2):", w_clf_v2[:5])

Max absolute difference in probabilities: 0.04932473990281333
Number of predictions that flipped: 0

Sample weight comparison (first 5 weights):
w_clf   (v1): [ 0.55298735 -0.30664479  0.02171244 -1.09392789  0.81807312]
w_clf_v2 (v2): [ 0.58955253 -0.29373532  0.01982683 -0.99463498  0.72563436]


## Summary

Both the Scikit-learn and from-scratch NumPy implementations gave nearly identical results, since they're solving the same underlying math. For **regression**, both got MAE ≈ 0.108, RMSE ≈ 0.149, and R² ≈ 0.168. For **classification**, both got Accuracy = 77.5%, Precision = 78.1%, Recall = 96.6%, and F1 = 0.864. Adding **L2 regularization** to the manual model slightly improved regression (R² rose to 0.169, MAE/RMSE dropped a touch) and changed the classification model's internal confidence by up to ~5 percentage points, though it didn't flip any final predictions. On timing, the manual version was often **faster** (e.g. 0.011s vs 0.046s to train the regression model) since it skips Scikit-learn's extra overhead, but Scikit-learn caught up on classification (0.080s vs 0.151s) thanks to its more efficient optimizer.